# Day 2: Hands-on with the End-to-End Machine Learning Cycle

Welcome! In this tutorial, you will see the **entire Machine Learning lifecycle** in action on the classic **Titanic Dataset** using simple Pandas and Scikit-Learn functions.

### The 6 Stages of the ML Lifecycle:
1. **Problem Formulation & Data Loading**: Target = `survived` (1 = Survived, 0 = Died).
2. **Exploratory Data Analysis (EDA)**: Inspect data, missing values, and summary stats.
3. **Simple Data Preprocessing**: Fill missing values, One-Hot Encode text columns, and Split Train/Test sets.
4. **Model Training**: Train Baseline, Logistic Regression, and Decision Tree models.
5. **Model Evaluation**: Calculate Accuracy, Precision, Recall, and Confusion Matrix.
6. **Model Persistence & Inference**: Save model via `joblib` and predict survival for a new passenger.

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

## Stage 1: Load Data & Define Problem

In [ ]:
df = pd.read_csv('../data/titanic.csv')
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## Stage 2: Exploratory Data Analysis (EDA)

In [ ]:
print('Missing Values:')
print(df.isnull().sum())
print(f'Overall Survival Rate: {df["survived"].mean()*100:.1f}%')

## Stage 3: Simple Preprocessing & Train/Test Split

In [ ]:
# 1. Fill missing values
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

# 2. One-Hot Encode text columns
df_encoded = pd.get_dummies(df, columns=['sex', 'embarked'], drop_first=True, dtype=int)

# 3. Separate X and y
X = df_encoded.drop(columns=['survived'])
y = df_encoded['survived']

# 4. Split Train/Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Stage 4: Model Training

In [ ]:
models = {
    'Baseline (Most Frequent)': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(max_depth=4, random_state=42)
}

trained_models = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model

## Stage 5: Evaluation

In [ ]:
for name, model in trained_models.items():
    preds = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, preds)
    print(f"{name}: Accuracy = {acc*100:.1f}%")

## Stage 6: Model Saving & Inference

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/titanic_scaler.joblib')
joblib.dump(trained_models['Logistic Regression'], '../models/titanic_model.joblib')

loaded_scaler = joblib.load('../models/titanic_scaler.joblib')
loaded_model = joblib.load('../models/titanic_model.joblib')

sample_passenger = X_test.iloc[[0]]
sample_scaled = loaded_scaler.transform(sample_passenger)
pred = loaded_model.predict(sample_scaled)[0]
print(f"Prediction for sample passenger: {'Survived' if pred==1 else 'Did not survive'}")